# Phase 2 — Person Detection: Model Exploration & Comparison

This notebook benchmarks YOLO11 models (`yolo11n`, `yolo11s`, `yolo11m`) on CUDA,
inspects detection confidence filtering, and visualizes annotated output frames.


In [ ]:
import sys
from pathlib import Path
import torch
import cv2
import matplotlib.pyplot as plt

# Add scripts to sys.path
SCRIPTS_DIR = Path.cwd().parent / "scripts"
sys.path.insert(0, str(SCRIPTS_DIR))
INSPECTION_DIR = Path.cwd().parents[1] / "01_video_inspection" / "scripts"
sys.path.insert(0, str(INSPECTION_DIR))

from detect_people import PersonDetector, draw_detections, select_default_device
from video_source import VideoSource

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", torch.cuda.get_device_properties(0).total_memory / (1024**3))


## 1. Test Single Frame Inference
We load a frame from the fallback test footage and detect people using YOLO11m baseline.


In [ ]:
video_path = Path.cwd().parents[1].parent / "data" / "fallback_clips" / "people-detection.mp4"
with VideoSource(video_path) as src:
    frame = src.read_frame(100)

detector = PersonDetector(model_name="yolo11m.pt", confidence_threshold=0.4)
detections = detector.detect(frame)
print(f"Detected {len(detections)} person(s) on frame 100")

annotated = draw_detections(frame, detections)
plt.figure(figsize=(10, 6))
plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.title(f"YOLO11m Detection (conf >= 0.4): {len(detections)} persons")
plt.show()


## 2. Compare Models (n vs s vs m)
Benchmark speed (FPS) and detection counts across model sizes.


In [ ]:
from compare_models import compare_models, print_comparison_table

results = compare_models(
    video_path,
    models=("yolo11n.pt", "yolo11s.pt", "yolo11m.pt"),
    confidence_threshold=0.4,
    max_frames=150,
)
print_comparison_table(results)
